In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


In [3]:
datasets_dir = Path('../Datasets/Embeddings')

models = {
    'BERT': 'extreme_pole_embeddings_bert.csv',
    'RoBERTa': 'extreme_pole_embeddings_roberta.csv',
    'NeoBERT': 'extreme_pole_embeddings_neobert.csv',
    'ModernBERT': 'extreme_pole_embeddings_modernbert.csv'
}


In [5]:
def load_embeddings(filepath):
    df = pd.read_csv(filepath)
    emb_cols = [col for col in df.columns if col.startswith('emb_')]
    return df, emb_cols


In [7]:
def calculate_mean_embeddings(df, emb_cols):
    results = {}
    
    for dimension in ['V', 'A', 'D']:
        for pole in ['pos', 'neg']:
            mask = (df['V/A/D'] == dimension) & (df['pos/neg'] == pole)
            mean_emb = df.loc[mask, emb_cols].mean().values
            results[f'{pole}_{dimension}'] = mean_emb
    
    return results


In [9]:
def calculate_direction_vectors(mean_embeddings):
    directions = {}
    
    for dimension in ['V', 'A', 'D']:
        pos_vec = mean_embeddings[f'pos_{dimension}']
        neg_vec = mean_embeddings[f'neg_{dimension}']
        direction = pos_vec - neg_vec
        directions[dimension] = direction
    
    return directions


In [11]:
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm1 = np.linalg.norm(vec1)
    norm2 = np.linalg.norm(vec2)
    return dot_product / (norm1 * norm2)


In [13]:
def calculate_orthogonality_metrics(directions):
    v_vec = directions['V']
    a_vec = directions['A']
    d_vec = directions['D']
    
    cos_va = cosine_similarity(v_vec, a_vec)
    cos_vd = cosine_similarity(v_vec, d_vec)
    cos_ad = cosine_similarity(a_vec, d_vec)
    
    abs_cos_va = abs(cos_va)
    abs_cos_vd = abs(cos_vd)
    abs_cos_ad = abs(cos_ad)
    
    mean_abs_cos = (abs_cos_va + abs_cos_vd + abs_cos_ad) / 3
    
    return {
        'cos_V_A': cos_va,
        'cos_V_D': cos_vd,
        'cos_A_D': cos_ad,
        'abs_cos_V_A': abs_cos_va,
        'abs_cos_V_D': abs_cos_vd,
        'abs_cos_A_D': abs_cos_ad,
        'mean_abs_cos': mean_abs_cos
    }


In [15]:
all_results = {}

for model_name, filename in models.items():
    print(f'Processing {model_name}...')
    
    filepath = datasets_dir / filename
    df, emb_cols = load_embeddings(filepath)
    
    mean_embeddings = calculate_mean_embeddings(df, emb_cols)
    
    directions = calculate_direction_vectors(mean_embeddings)
    
    orthogonality = calculate_orthogonality_metrics(directions)
    
    all_results[model_name] = {
        'mean_embeddings': mean_embeddings,
        'directions': directions,
        'orthogonality': orthogonality
    }
    
    print(f'{model_name} complete\\n')


Processing BERT...
BERT complete\n
Processing RoBERTa...
RoBERTa complete\n
Processing NeoBERT...
NeoBERT complete\n
Processing ModernBERT...
ModernBERT complete\n


In [17]:
orthogonality_df = pd.DataFrame({
    model: results['orthogonality']
    for model, results in all_results.items()
}).T

print('Orthogonality Metrics (lower absolute cosine similarity is better):')
print(orthogonality_df.round(4))
print()

best_model = orthogonality_df['mean_abs_cos'].idxmin()
print(f'Best orthogonality: {best_model}')
print(f'Mean absolute cosine similarity: {orthogonality_df.loc[best_model, "mean_abs_cos"]:.4f}')


Orthogonality Metrics (lower absolute cosine similarity is better):
            cos_V_A  cos_V_D  cos_A_D  abs_cos_V_A  abs_cos_V_D  abs_cos_A_D  \
BERT        -0.1008   0.2354   0.2660       0.1008       0.2354       0.2660   
RoBERTa     -0.0513   0.2413   0.2374       0.0513       0.2413       0.2374   
NeoBERT     -0.2694   0.1213   0.4273       0.2694       0.1213       0.4273   
ModernBERT  -0.1068   0.1981   0.0783       0.1068       0.1981       0.0783   

            mean_abs_cos  
BERT              0.2008  
RoBERTa           0.1767  
NeoBERT           0.2727  
ModernBERT        0.1278  

Best orthogonality: ModernBERT
Mean absolute cosine similarity: 0.1278


In [19]:
output_rows = []

for model_name, results in all_results.items():
    mean_embeddings = results['mean_embeddings']
    directions = results['directions']
    orthogonality = results['orthogonality']
    
    for key, vector in mean_embeddings.items():
        row = {
            'model': model_name,
            'vector_type': 'mean_embedding',
            'vector_name': key
        }
        for i, val in enumerate(vector):
            row[f'dim_{i}'] = val
        output_rows.append(row)
    
    for dimension, vector in directions.items():
        row = {
            'model': model_name,
            'vector_type': 'direction',
            'vector_name': f'dir_{dimension}'
        }
        for i, val in enumerate(vector):
            row[f'dim_{i}'] = val
        output_rows.append(row)
    
    row = {
        'model': model_name,
        'vector_type': 'orthogonality',
        'vector_name': 'metrics',
        'dim_0': orthogonality['cos_V_A'],
        'dim_1': orthogonality['cos_V_D'],
        'dim_2': orthogonality['cos_A_D'],
        'dim_3': orthogonality['abs_cos_V_A'],
        'dim_4': orthogonality['abs_cos_V_D'],
        'dim_5': orthogonality['abs_cos_A_D'],
        'dim_6': orthogonality['mean_abs_cos']
    }
    output_rows.append(row)

output_df = pd.DataFrame(output_rows)
output_path = '../Datasets/orthogonality_analysis_results.csv'
output_df.to_csv(output_path, index=False)

print(f'Results saved to {output_path}')
print(f'Total rows: {len(output_df)}')


Results saved to ../Datasets/orthogonality_analysis_results.csv
Total rows: 40


In [21]:
print('Summary:')
print(f'Total models analyzed: {len(models)}')
print(f'Best model for orthogonality: {best_model}')
print('\\nOrthogonality ranking (lower is better):')
ranking = orthogonality_df.sort_values('mean_abs_cos')[['mean_abs_cos']]
for i, (model, row) in enumerate(ranking.iterrows(), 1):
    print(f'{i}. {model}: {row["mean_abs_cos"]:.4f}')


Summary:
Total models analyzed: 4
Best model for orthogonality: ModernBERT
\nOrthogonality ranking (lower is better):
1. ModernBERT: 0.1278
2. RoBERTa: 0.1767
3. BERT: 0.2008
4. NeoBERT: 0.2727
